In [6]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [7]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

Handling JSON data is one of PySpark's core strengths, especially when dealing with the nested structures common in modern web APIs.  

To demonstrate this, we will use a sample dataset representing a "Product Order" where information is stored as a JSON string within a DataFrame.

# The Sample Dataset

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Sample data: A list of orders where 'details' is a JSON string
data = [
    (101, '{"customer": "Alice", "items": [{"id": 1, "p": "Laptop"}, {"id": 2, "p": "Mouse"}]}'),
    (102, '{"customer": "Bob", "items": [{"id": 3, "p": "Monitor"}]}'),
    (103, '{"customer": "Charlie", "items": []}')
]

df = spark.createDataFrame(data, ["order_id", "details"])
df.show(truncate=False)

ModuleNotFoundError: No module named 'pyspark'

### 1. from_json: Parsing JSON Strings into Structs

This is the most powerful API for JSON. It converts a string column into a complex Spark type (Struct, Array, or Map). You **must** provide a schema.  

**Explanation:** Use this when you want to "materialize" the entire JSON structure into columns you can interact with.

In [4]:
# Define the schema matching the JSON structure
schema = StructType([
    StructField("customer", StringType()),
    StructField("items", ArrayType(StructType([
        StructField("id", IntegerType()),
        StructField("p", StringType())
    ])))
])

# Parse the string column
parsed_df = df.withColumn("parsed_details", from_json(col("details"), schema))

parsed_df.select("order_id", "parsed_details.customer", "parsed_details.items").show(truncate=False)

+--------+--------+-------------------------+
|order_id|customer|items                    |
+--------+--------+-------------------------+
|101     |Alice   |[[1, Laptop], [2, Mouse]]|
|102     |Bob     |[[3, Monitor]]           |
|103     |Charlie |[]                       |
+--------+--------+-------------------------+



### 2. get_json_object: Quick Extraction

If you only need one or two specific values from a JSON string, `get_json_object` is much faster than defining a whole schema. It uses **JSONPath** syntax.

**Explanation:** Ideal for "cherry-picking" specific fields without the overhead of parsing everything.

In [5]:
# Extract only the customer name
# '$' represents the root; '.customer' drills down
simple_df = df.withColumn("cust_name", get_json_object(col("details"), "$.customer"))

simple_df.select("order_id", "cust_name").show()

+--------+---------+
|order_id|cust_name|
+--------+---------+
|     101|    Alice|
|     102|      Bob|
|     103|  Charlie|
+--------+---------+



### 3. explode: Handling Nested Arrays

Often, JSON contains arrays (like our `items` list). `explode` creates a new row for every element in that array.

**Explanation:** Essential for flattening data for analysis (e.g., counting total items sold across all orders).

In [6]:
# First parse the JSON, then explode the 'items' array
flattened_df = parsed_df.withColumn("item", explode(col("parsed_details.items")))

# Select nested fields from the exploded struct
flattened_df.select(
    "order_id", 
    "parsed_details.customer", 
    "item.id", 
    "item.p"
).show()

+--------+--------+---+-------+
|order_id|customer| id|      p|
+--------+--------+---+-------+
|     101|   Alice|  1| Laptop|
|     101|   Alice|  2|  Mouse|
|     102|     Bob|  3|Monitor|
+--------+--------+---+-------+



### 4. to_json: Converting Back to String

Sometimes you need to prepare data to be sent to a NoSQL database or a web hook. `to_json` converts a Struct or Array column back into a JSON-formatted string.

**Explanation:** Useful for "packaging" multiple columns into a single JSON blob.

In [7]:
# Pack order_id and customer back into a JSON string
json_output_df = parsed_df.select(
    to_json(struct("order_id", "parsed_details.customer")).alias("json_payload")
)

json_output_df.show(truncate=False)

+-------------------------------------+
|json_payload                         |
+-------------------------------------+
|{"order_id":101,"customer":"Alice"}  |
|{"order_id":102,"customer":"Bob"}    |
|{"order_id":103,"customer":"Charlie"}|
+-------------------------------------+



### 5. schema_of_json: Auto-detecting Schema

If your JSON is massive and complex, writing a manual StructType is painful. `schema_of_json` can infer the schema from a single example string.

**Explanation:** Use this to generate the DDL string required for the from_json function automatically.

In [8]:
# Get the schema of a sample string
json_sample = '{"customer": "Alice", "items": [{"id": 1, "p": "Laptop"}]}'
inferred_schema = schema_of_json(lit(json_sample))

# Use the inferred schema to parse the column
df.select(from_json(col("details"), inferred_schema)).printSchema()

root
 |-- jsontostructs(details): struct (nullable = true)
 |    |-- customer: string (nullable = true)
 |    |-- items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- p: string (nullable = true)



<img src="images/pyspark_json_api_summary.jpg" align = "left">

In [27]:
# Need to work on this problem
# Ref - https://github.com/tariqulislam/spark-json-example/blob/main/src/test_remote_json_api.py

import requests
from pprint import pprint
from pyspark.sql.functions import *
from pyspark.sql.types import *

import pyspark.sql.functions as F

# Fetch data from a public API
url = 'https://dummyjson.com/products'
response = requests.get(url)

# Convert response to json
json_data = response.json()


pprint(json_data)

{'limit': 30,
 'products': [{'availabilityStatus': 'In Stock',
               'brand': 'Essence',
               'category': 'beauty',
               'description': 'The Essence Mascara Lash Princess is a popular '
                              'mascara known for its volumizing and '
                              'lengthening effects. Achieve dramatic lashes '
                              'with this long-lasting and cruelty-free '
                              'formula.',
               'dimensions': {'depth': 22.99, 'height': 13.08, 'width': 15.14},
               'discountPercentage': 10.48,
               'id': 1,
               'images': ['https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp'],
               'meta': {'barcode': '5784719087687',
                        'createdAt': '2025-04-30T09:41:02.053Z',
                        'qrCode': 'https://cdn.dummyjson.com/public/qr-code.png',
                        'updatedAt': '2025-04-30T09:41:02.05

                              'baking, or pan-searing.',
               'dimensions': {'depth': 11.27, 'height': 26.31, 'width': 14.95},
               'discountPercentage': 4.23,
               'id': 24,
               'images': ['https://cdn.dummyjson.com/product-images/groceries/fish-steak/1.webp'],
               'meta': {'barcode': '9595036192098',
                        'createdAt': '2025-04-30T09:41:02.053Z',
                        'qrCode': 'https://cdn.dummyjson.com/public/qr-code.png',
                        'updatedAt': '2025-04-30T09:41:02.053Z'},
               'minimumOrderQuantity': 50,
               'price': 14.99,
               'rating': 3.78,
               'returnPolicy': '60 days return policy',
               'reviews': [{'comment': 'Would not buy again!',
                            'date': '2025-04-30T09:41:02.053Z',
                            'rating': 2,
                            'reviewerEmail': 'caleb.perkins@x.dummyjson.com',
                        

In [33]:
rdd = spark.sparkContext.parallelize([json_data])
df = spark.read.option("allowSingleQuotes", "true").json(rdd)
# df.show()
df.printSchema()

df2 = df.withColumn("products", F.explode("products"))
# df2.show()
# df3=df2.select("products.id","products.brand",\
#          *[F.col('products.images').getItem(i)\
#    .alias(f'img{i+1}') for i in range(0,len('products.images'))])

df3=df2.select("products.id","products.brand", F.col('products.images'))
# df3.printSchema()
df3.show(truncate = False)

# inferred_schema = schema_of_json(col(json_data))
# print(inferred_schema)

root
 |-- limit: long (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- availabilityStatus: string (nullable = true)
 |    |    |-- brand: string (nullable = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- dimensions: struct (nullable = true)
 |    |    |    |-- depth: double (nullable = true)
 |    |    |    |-- height: double (nullable = true)
 |    |    |    |-- width: double (nullable = true)
 |    |    |-- discountPercentage: double (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- images: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- meta: struct (nullable = true)
 |    |    |    |-- barcode: string (nullable = true)
 |    |    |    |-- createdAt: string (nullable = true)
 |    |    |    |-- qrCode: string (nullable = true)
 |    |    |    |-- updatedAt: string 

In [16]:
from pyspark.sql.types import (
    StructType, 
    StructField, 
    StringType, 
    IntegerType, 
    DoubleType, 
    ArrayType
)

# 1. Define the Review Schema (nested inside Product)
review_schema = StructType([
    StructField("comment", StringType(), True),
    StructField("date", StringType(), True),
    StructField("rating", IntegerType(), True),
    StructField("reviewerEmail", StringType(), True),
    StructField("reviewerName", StringType(), True)
])

# 2. Define the Dimensions Schema (nested inside Product)
dimensions_schema = StructType([
    StructField("depth", DoubleType(), True),
    StructField("height", DoubleType(), True),
    StructField("width", DoubleType(), True)
])

# 3. Define the Meta Schema (nested inside Product)
meta_schema = StructType([
    StructField("barcode", StringType(), True),
    StructField("createdAt", StringType(), True),
    StructField("qrCode", StringType(), True),
    StructField("updatedAt", StringType(), True)
])

# 4. Define the Product Schema (The main repeatable object)
product_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("discountPercentage", DoubleType(), True),
    StructField("rating", DoubleType(), True),
    StructField("stock", IntegerType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("brand", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("weight", IntegerType(), True),
    StructField("dimensions", dimensions_schema, True),
    StructField("warrantyInformation", StringType(), True),
    StructField("shippingInformation", StringType(), True),
    StructField("availabilityStatus", StringType(), True),
    StructField("reviews", ArrayType(review_schema), True),
    StructField("returnPolicy", StringType(), True),
    StructField("minimumOrderQuantity", IntegerType(), True),
    # Note: Meta is a struct, not an array
    StructField("meta", meta_schema, True),
    StructField("images", ArrayType(StringType()), True),
    StructField("thumbnail", StringType(), True)
])

# 5. Final Root Schema
final_schema = StructType([
    StructField("products", ArrayType(product_schema), True),
    StructField("total", IntegerType(), True),
    StructField("skip", IntegerType(), True),
    StructField("limit", IntegerType(), True)
])

In [18]:
from pyspark.sql.functions import from_json, col

df = spark.createDataFrame(json_data, schema=final_schema)
df.printSchema()

# df_parsed = df.withColumn("json_data", from_json(col("raw_data"), final_schema))

# To access the first product's title:
# df_parsed.select("json_data.products[0].title").show()

TypeError: StructType can not accept object 'products' in type <class 'str'>